In [33]:
# load data
import pandas as pd

df = pd.read_csv('valorant-stats.csv')
df

,Name,Weapon Type,Price,Fire Rate,Wall Penetration,Magazine Capacity,HDMG_0,BDMG_0,LDMG_0,HDMG_1,BDMG_1,LDMG_1,HDMG_2,BDMG_2,LDMG_2
0,Classic,Sidearm,0,6.75,Low,12,78,26,22,78,26,22,66,22,18
1,Shorty,Sidearm,200,3.30,Low,2,36,12,10,24,8,6,9,3,2
2,Frenzy,Sidearm,400,10.00,Low,13,78,26,22,63,21,17,63,21,17
3,Ghost,Sidearm,500,6.75,Medium,15,105,33,26,88,25,21,88,25,21
4,Sheriff,Sidearm,800,4.00,High,6,160,55,47,160,55,47,145,50,43
5,Stinger,SMG,1000,18.00,Low,20,67,27,23,62,25,21,62,25,21
6,Spectre,SMG,1600,13.33,Medium,30,78,26,22,66,22,18,66,22,18
7,Bulldog,Rifle,2100,9.15,Medium,24,116,35,30,116,35,30,116,35,30
8,Guardian,Rifle,2500,6.50,Medium,12,195,65,49,195,65,49,195,65,49
9,Phantom,Rifle,2900,11.00,Medium,30,156,39,33,140,35,30,124,31,26


In [43]:
# EDA
print(f'Cek 5 data teratas \n {df.head()} \n')

print(f'Cek tipe data \n {df.dtypes} \n')

print(f'Cek missing values \n {df.isnull().sum()} \n')

print(f'Cek duplicate data \n {df.duplicated().sum()}')

Cek 5 data teratas 
   Weapon Type  Price  Fire Rate  Wall Penetration  Magazine Capacity  HDMG_0  \
0     Sidearm      0       6.75                 1                 12      78   
1     Sidearm    200       3.30                 1                  2      36   
2     Sidearm    400      10.00                 1                 13      78   
3     Sidearm    500       6.75                 2                 15     105   
4     Sidearm    800       4.00                 0                  6     160   

   BDMG_0  LDMG_0  HDMG_1  BDMG_1  LDMG_1  HDMG_2  BDMG_2  LDMG_2  
0      26      22      78      26      22      66      22      18  
1      12      10      24       8       6       9       3       2  
2      26      22      63      21      17      63      21      17  
3      33      26      88      25      21      88      25      21  
4      55      47     160      55      47     145      50      43   

Cek tipe data 
 Weapon Type              str
Price                  int64
Fire Rate     

In [44]:
# cek outlier
numerical_columns = df.select_dtypes(include='number').columns

outlier_summary = []

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    outlier_summary.append({
        'kolom': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'batas_bawah': lower_bound,
        'batas_atas': upper_bound,
        'jumlah_outlier': len(outliers),
        'persen_outlier': round(len(outliers) / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,kolom,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_outlier,persen_outlier
0,Price,800.0,2500.0,1700.0,-1750.00,5050.00,0,0.00
1,Fire Rate,3.5,10.0,6.5,-6.25,19.75,0,0.00
2,Wall Penetration,1.0,2.0,1.0,-0.50,3.50,0,0.00
3,Magazine Capacity,6.0,25.0,19.0,-22.50,53.50,1,5.88
4,HDMG_0,72.0,156.0,84.0,-54.00,282.00,0,0.00
5,BDMG_0,26.0,39.0,13.0,6.50,58.50,3,17.65
6,LDMG_0,22.0,33.0,11.0,5.50,49.50,2,11.76
7,HDMG_1,63.0,156.0,93.0,-76.50,295.50,0,0.00
8,BDMG_1,22.0,39.0,17.0,-3.50,64.50,3,17.65
9,LDMG_1,18.0,33.0,15.0,-4.50,55.50,2,11.76


In [45]:
# handle outlier
df_capped = df.copy()

for kolom in numerical_columns:

    Q1 = df_capped[kolom].quantile(0.25)
    Q3 = df_capped[kolom].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    df_capped[kolom] = df_capped[kolom].clip(
        lower=lower,
        upper=upper
    )

In [46]:
# encoding
# (Ordinal Encoding)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Wall Penetration'] = le.fit_transform(df['Wall Penetration'])

# One Hot Encoding
df_encoded = pd.get_dummies(
    df,
    columns=['Weapon Type'],
    dtype=int
)

In [47]:
# split data
x = df.drop(columns=['Price'])
y = df['Price']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [48]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(max_depth=2, random_state=100)
rf.fit(x_train, y_train)

ValueError: could not convert string to float: 'Sniper'